# Declared TPM feature domains

This walkthrough demonstrates the U-Bounds extension using the committed schema-v3 TPM artifact. It keeps four meanings separate:

1. a feature's mathematical **decision domain** or **distribution support**;
2. the physical values actually observed in **training support**;
3. the physical-to-model-coordinate **normalization** and its provenance; and
4. the finite **compiler approximation domain** selected for one encoding.

The important point is not merely that OEL stores a pair of bounds. It records what each interval means and prevents one meaning from silently substituting for another.


## Reproducible execution

Run this notebook from the OEL repository root. Install the optional visualization and kernel tools with `uv sync --group notebook`, then start it with:

```bash
uv run jupyter lab examples/tpm_planning/declared_feature_domains_walkthrough.ipynb
```

The notebook is offline and CPU-only. It deliberately reloads the reviewed, digest-checked v3 fixture instead of refitting SPFlow. The canonical regeneration and verification path remains `run_declared_domains_pipeline.py`.


In [1]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from open_energy_lab.tpm import (
    TPMApproximationDomain,
    TPMQueryRequest,
    build_compiled_spflow_decision_link_model,
    compile_spflow_model_to_mio,
    evaluate_compiled_spflow_probability,
    load_tpm_artifact,
    normalize_training_data,
    query_tpm_artifact,
)
from open_energy_lab.tpm.optional_spflow import load_spflow_payload
from open_energy_lab.tpm.query import marginalize_spflow_model

working_directory = Path.cwd().resolve()
REPOSITORY_ROOT = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src/open_energy_lab").is_dir()
    ),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Run this checked-in notebook from within the OEL repository.")

ARTIFACT_PATH = REPOSITORY_ROOT / "tests/fixtures/tpm/tpm_artifact_domains_v3.json"
assert ARTIFACT_PATH.is_file()


## 1. Inspect persisted meanings

The fitted artifact contains a finite decision domain for $x$, exact declared binary and categorical domains, explicitly unbounded demand support, observed training support, and the corresponding transforms. The event target is generated internally and remains distinct from the caller declarations.


In [2]:
artifact = load_tpm_artifact(ARTIFACT_PATH)
assert artifact.schema_version == "v3"
feature_by_name = {spec.feature_name: spec for spec in artifact.feature_specs}

def mathematical_meaning(spec):
    domain = spec.mathematical_domain
    if domain.feature_kind == "continuous":
        endpoints = {"finite": f"[{domain.lower}, {domain.upper}]",
                     "lower-bounded": f"[{domain.lower}, +unbounded)",
                     "upper-bounded": f"(-unbounded, {domain.upper}]",
                     "unbounded": "(-unbounded, +unbounded)",
                     "unknown": "unknown"}[domain.shape]
        return f"{domain.purpose}: {endpoints}"
    return f"{domain.purpose}: {domain.ordering}"

def observed_support(spec):
    support = spec.training_support
    if support.feature_kind == "continuous":
        return f"[{support.minimum}, {support.maximum}]"
    return str(support.observed_values)

semantic_summary = pd.DataFrame(
    [
        {
            "feature": spec.feature_name,
            "role": spec.feature_role,
            "mathematical meaning": mathematical_meaning(spec),
            "observed training support": observed_support(spec),
            "normalization": spec.normalization.kind,
            "normalization origin": spec.normalization.origin,
        }
        for spec in artifact.feature_specs
    ]
).set_index("feature")
display(semantic_summary)


## 2. Training extrema no longer define the coordinate system

Both small datasets below declare the same mathematical decision domain, $40 \leq x \leq 160$, but observe different subsets. The shared probe $x=100$ must map to $0.5$ in both cases. This is the core correction over using sampled minima and maxima as if they were theoretical bounds.


In [3]:
declared_x = feature_by_name["x"].mathematical_domain
wide_observations = normalize_training_data(
    ({"x": 60.0}, {"x": 140.0}),
    feature_domains={"x": declared_x},
    feature_roles={"x": "decision"},
)
narrow_observations = normalize_training_data(
    ({"x": 80.0}, {"x": 120.0}),
    feature_domains={"x": declared_x},
    feature_roles={"x": "decision"},
)
wide_x = wide_observations.feature_schema[0]
narrow_x = narrow_observations.feature_schema[0]

normalization_comparison = pd.DataFrame(
    {
        "physical x": [40.0, 80.0, 100.0, 120.0, 160.0],
        "coordinate; observed [60, 140]": [
            wide_x.normalize_value(value) for value in (40.0, 80.0, 100.0, 120.0, 160.0)
        ],
        "coordinate; observed [80, 120]": [
            narrow_x.normalize_value(value) for value in (40.0, 80.0, 100.0, 120.0, 160.0)
        ],
    }
)
assert all(
    math.isclose(left, right, abs_tol=1e-12)
    for left, right in zip(
        normalization_comparison.iloc[:, 1],
        normalization_comparison.iloc[:, 2],
        strict=True,
    )
)
display(normalization_comparison)


## 3. Unbounded and discrete declarations carry their own semantics

Demand remains explicitly unbounded even though a finite sample was fitted, so it uses identity normalization and retains physical units. Likewise, the complete category `priority` is valid because it was declared, even though it was absent from training. No synthetic observation is added.


In [4]:
demand_spec = feature_by_name["demand"]
tier_spec = feature_by_name["service_tier"]
x_spec = feature_by_name["x"]

relevant_cases = pd.DataFrame(
    [
        {
            "case": "unbounded demand probe",
            "physical value": 250.0,
            "declared valid": True,
            "observed in training": False,
            "model coordinate": demand_spec.normalize_value(250.0),
        },
        {
            "case": "declared but unobserved category",
            "physical value": "priority",
            "declared valid": "priority" in tier_spec.mathematical_domain.values,
            "observed in training": "priority" in tier_spec.training_support.observed_values,
            "model coordinate": tier_spec.normalize_value("priority"),
        },
        {
            "case": "decision endpoint outside observations",
            "physical value": 40.0,
            "declared valid": True,
            "observed in training": x_spec.training_support.minimum <= 40.0 <= x_spec.training_support.maximum,
            "model coordinate": x_spec.normalize_value(40.0),
        },
    ]
)
assert demand_spec.mathematical_domain.shape == "unbounded"
assert demand_spec.normalization.kind == "identity"
assert relevant_cases.loc[1, "declared valid"] and not relevant_cases.loc[1, "observed in training"]
display(relevant_cases.set_index("case"))


## 4. Native queries validate declarations, not sample coverage

The next queries are accepted at the declared endpoints and at an unobserved declared category. Acceptance means that the evidence is mathematically admissible; it does not claim that sparse training coverage makes the estimate accurate. A value outside the declared decision domain fails before SPFlow evaluation.


In [5]:
native_query_inputs = {
    "lower endpoint outside training support": {
        "x": 40.0, "activation": "off", "service_tier": "economy"
    },
    "upper endpoint outside training support": {
        "x": 160.0, "activation": "on", "service_tier": "standard"
    },
    "declared unobserved category": {
        "x": 100.0, "activation": "on", "service_tier": "priority"
    },
}
native_query_results = []
for case_name, assignments in native_query_inputs.items():
    result = query_tpm_artifact(
        artifact,
        TPMQueryRequest(decision_assignments=assignments),
    )
    native_query_results.append(
        {
            "case": case_name,
            "source": result["source"],
            "conditional probability": result["conditional_probability"],
        }
    )
native_query_table = pd.DataFrame(native_query_results).set_index("case")
display(native_query_table)

try:
    query_tpm_artifact(
        artifact,
        TPMQueryRequest(
            decision_assignments={
                "x": 39.0, "activation": "off", "service_tier": "economy"
            }
        ),
    )
except ValueError as error:
    rejected_native_query = {
        "exception": type(error).__name__,
        "reason": getattr(error, "reason", None),
        "semantic source": getattr(error, "semantic_source", None),
        "message": str(error),
    }
else:
    raise AssertionError("x=39 must be rejected by the declared decision domain")
assert rejected_native_query["reason"] == "decision_domain_violation"
display(pd.Series(rejected_native_query, name="expected rejection"))


## 5. Compilation requires a separate finite validity interval

The SPFlow decision feature has mathematical domain $[40,160]$, but this particular piecewise approximation is deliberately qualified only on $[60,140]$. OEL derives the model-coordinate interval $[1/6,5/6]$ through the artifact's declared normalization. It does not renormalize the subset to $[0,1]$, and it does not borrow observed extrema implicitly. Demand is structurally marginalized, so its unbounded support does not acquire an artificial compiler interval.


In [6]:
runtime = load_spflow_payload(artifact)
assert runtime is not None
demand_index = next(
    index for index, spec in enumerate(artifact.feature_specs) if spec.feature_name == "demand"
)
model_without_demand = marginalize_spflow_model(runtime["model"], [demand_index])
compiled = compile_spflow_model_to_mio(
    model_without_demand,
    fitted_artifact=artifact,
    approximation_domains=(
        TPMApproximationDomain(
            feature_name="x",
            lower=60.0,
            upper=140.0,
            source="declared-feature-domains-walkthrough",
        ),
    ),
)
assert compiled["schema_version"] == "v2"
assert len(compiled["compiled_validity"]) == 1
compiled_validity = compiled["compiled_validity"][0]
validity_table = pd.Series(
    {
        "feature": compiled_validity["feature_name"],
        "physical interval": compiled_validity["physical_interval"],
        "model-coordinate interval": compiled_validity["model_interval"],
        "physical breakpoints": compiled_validity["physical_breakpoints"],
        "model breakpoints": compiled_validity["model_breakpoints"],
        "source": compiled_validity["source"],
        "encoding": compiled_validity["encoding"],
    },
    name="persisted compiled validity",
)
display(validity_table)


In [7]:
fig, (interval_ax, coordinate_ax) = plt.subplots(1, 2, figsize=(12, 4.2))
intervals = [
    ("declared decision domain", 40.0, 160.0, "#0072B2"),
    ("observed training support", 60.0, 140.0, "#E69F00"),
    ("compiler validity", 60.0, 140.0, "#009E73"),
]
for row, (label, lower, upper, color) in enumerate(intervals):
    interval_ax.hlines(row, lower, upper, color=color, linewidth=8)
    interval_ax.scatter([lower, upper], [row, row], color=color, s=55, zorder=3)
interval_ax.set_yticks(range(len(intervals)), [item[0] for item in intervals])
interval_ax.set_xlim(35, 165)
interval_ax.set_xlabel("physical x")
interval_ax.set_title("Different intervals may coincide without sharing meaning")
interval_ax.grid(axis="x", alpha=0.25)

physical_axis = [40.0, 60.0, 100.0, 140.0, 160.0]
model_axis = [x_spec.normalize_value(value) for value in physical_axis]
coordinate_ax.plot(physical_axis, model_axis, marker="o", color="#0072B2")
coordinate_ax.plot(physical_axis[1:4], model_axis[1:4], linewidth=7, alpha=0.45, color="#009E73")
coordinate_ax.set_xlabel("physical x")
coordinate_ax.set_ylabel("model coordinate")
coordinate_ax.set_title("Compiler subset keeps the fitted coordinate system")
coordinate_ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## 6. Physical and model-coordinate evaluation agree inside validity

Compiled-v2 accepts either physical evidence or model-coordinate evidence. Both paths are checked against the persisted validity record. The numeric value below is the compiled graph evaluation for the selected evidence; this notebook uses it to demonstrate coordinate parity, not as a benchmark chance-constraint result.


In [8]:
compiled_evaluations = []
for physical_x in (60.0, 100.0, 140.0):
    model_x = x_spec.normalize_value(physical_x)
    physical_probability = evaluate_compiled_spflow_probability(
        compiled,
        physical_evidence={
            "x": physical_x,
            "activation": "off",
            "service_tier": "economy",
            "demand_above_100": 0.0,
        },
    )
    coordinate_probability = evaluate_compiled_spflow_probability(
        compiled,
        model_coordinate_evidence={
            "x": model_x,
            "activation": 0.0,
            "service_tier": 0.0,
            "demand_above_100": 0.0,
        },
    )
    assert math.isclose(physical_probability, coordinate_probability, abs_tol=1e-12)
    compiled_evaluations.append(
        {
            "physical x": physical_x,
            "model x": model_x,
            "physical-path value": physical_probability,
            "coordinate-path value": coordinate_probability,
        }
    )
compiled_evaluation_table = pd.DataFrame(compiled_evaluations).set_index("physical x")
display(compiled_evaluation_table)

compiled_rejections = []
for physical_x in (59.999, 140.001):
    try:
        evaluate_compiled_spflow_probability(
            compiled,
            physical_evidence={
                "x": physical_x,
                "activation": "off",
                "service_tier": "economy",
                "demand_above_100": 0.0,
            },
        )
    except ValueError as error:
        compiled_rejections.append({"physical x": physical_x, "message": str(error)})
    else:
        raise AssertionError(f"x={physical_x} must be outside compiler validity")
assert len(compiled_rejections) == 2
display(pd.DataFrame(compiled_rejections).set_index("physical x"))


## 7. Pyomo and HiGHS link physical decisions to model coordinates

The library creates a bounded physical variable, a bounded model-coordinate variable, and their affine equality. Fixing $x=100$ therefore forces the coordinate to $0.5$. Fixing $x=59$ is infeasible because it lies outside this compiled approximation's physical validity interval, even though it remains inside the broader mathematical decision domain.


In [9]:
import pyomo.environ as pyo

link_model = build_compiled_spflow_decision_link_model(
    pyo,
    compiled_payload=compiled,
    decision_features=("x",),
)
x_variables = link_model._open_energy_lab_decision_variables["x"]
x_variables["physical_var"].fix(100.0)
solver = pyo.SolverFactory("highs")
solver.options["threads"] = 1
inside_result = solver.solve(link_model)

outside_model = build_compiled_spflow_decision_link_model(
    pyo,
    compiled_payload=compiled,
    decision_features=("x",),
)
outside_model._open_energy_lab_decision_variables["x"]["physical_var"].fix(59.0)
outside_result = solver.solve(outside_model, load_solutions=False)

highs_link_evidence = pd.Series(
    {
        "inside termination": str(inside_result.solver.termination_condition),
        "physical x": pyo.value(x_variables["physical_var"]),
        "derived model coordinate": pyo.value(x_variables["model_coordinate_var"]),
        "outside termination": str(outside_result.solver.termination_condition),
    },
    name="HiGHS decision-link evidence",
)
assert str(inside_result.solver.termination_condition).lower() == "optimal"
assert math.isclose(highs_link_evidence["derived model coordinate"], 0.5, abs_tol=1e-9)
assert "infeasible" in str(outside_result.solver.termination_condition).lower()
display(highs_link_evidence)


## What transfers to the research benchmarks

- **Scalar diagnostic:** declare $x \in [40,160]$ while normal demand remains explicitly unbounded and marginalized.
- **Knapsack:** selections have exact binary domains; Gaussian weights remain unbounded unless the benchmark explicitly truncates them. Capacity is a coupled model constraint, not a coordinate bound.
- **Newsvendor:** quantities may declare $0 \leq q_i \leq U_i$ and activations remain binary. Shared budget and storage constraints stay in the caller's optimization model.

Training support remains coverage evidence in every case. A compiler interval is requested only for a retained continuous approximation and qualifies only that compiled result. Integer linking, conditional numerator/denominator embedding, and benchmark optimization policy are separate features.
